# URA Hackathon 2026 - Pipeline v6.0 (PaddleOCR detect + VietOCR recognize)

**v6.0:** PaddleOCR detect bounding boxes -> VietOCR (vgg_seq2seq) recognize crops -> bilingual post-process

**Merge:** have_train v5.0 + pipeline.ipynb ideas

**Scoring:** `0.40 x F1_brand + 0.35 x (1-CER) + 0.25 x F1_product`

**Pipeline:** PaddleOCR(det) -> VietOCR(rec) -> bilingual_post -> BRAND_RULES -> classifiers -> split


In [1]:
import os, sys, subprocess
from concurrent.futures import ThreadPoolExecutor

CPU_THREADS = max(1, os.cpu_count() or 2)
PIP_WORKERS = min(CPU_THREADS, 4)
print(f'CPU threads: {CPU_THREADS} | pip workers: {PIP_WORKERS}')

for _env in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_env, str(CPU_THREADS))

def _pip(*pkgs):
    subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs], check=True)

print('Installing packages (v6.0: PaddleOCR detect + VietOCR recognize)...')
with ThreadPoolExecutor(max_workers=PIP_WORKERS) as pool:
    jobs = [
        pool.submit(_pip, 'paddlepaddle'),
        pool.submit(_pip, 'tqdm'),
        pool.submit(_pip, 'scikit-learn'),
    ]
    for j in jobs: j.result()
_pip('paddleocr>=2.7,<3')
print('Installing VietOCR + underthesea (CPU mode)...')
_pip('vietocr')
_pip('underthesea')
_pip('Pillow==9.5.0')  # Force downgrade Pillow as the absolute LAST step
print('Install done (PaddleOCR detect + VietOCR recognize)')


CPU threads: 4 | pip workers: 4
Installing: paddlepaddle (CPU) + tqdm + scikit-learn ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.9 MB/s eta 0:00:00
Install done (CPU-only PaddleOCR)


In [2]:
import os, re, csv, time, warnings, unicodedata
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image, ImageEnhance, ImageFilter
from collections import Counter
warnings.filterwarnings('ignore')

RUNNING_ON_KAGGLE = True  # True = Kaggle, False = Local

# ---- Auto-discovery duong dan (robust voi cau truc dataset doi giua cac vong/phase) ----
def _find_first(root, names):
    """Tim de quy file theo danh sach ten uu tien (vd private_test.csv truoc test.csv)."""
    if root is None or not Path(root).exists():
        return None
    root = Path(root)
    for nm in names:
        hits = sorted(root.rglob(nm))
        if hits:
            return hits[0]
    return None

def _find_images_dir(root, prefer=('test',)):
    """Tra ve thu muc chua nhieu anh (de quy). Uu tien thu muc co tu khoa 'test'
    de tranh chon nham thu muc anh TRAIN (vốn co the nhieu anh hon)."""
    if root is None or not Path(root).exists():
        return None
    root = Path(root)
    items = []
    cands = [root] + [p for p in root.rglob('*') if p.is_dir()]
    for d in cands:
        try:
            n = sum(1 for p in d.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
        except Exception:
            n = 0
        if n > 0:
            items.append((d, n))
    if not items:
        return None
    pref = [t for t in items if any(k in str(t[0]).lower() for k in prefer)]
    pool = pref if pref else items
    return max(pool, key=lambda t: t[1])[0]

if RUNNING_ON_KAGGLE:
    _roots = [
        Path('/kaggle/input/competitions/the-2nd-ura-hackathon'),
        Path('/kaggle/input/the-2nd-ura-hackathon'),
    ]
    COMP_ROOT = next((p for p in _roots if p.exists()), _roots[0])
    WORK_DIR  = Path('/kaggle/working')
else:
    COMP_ROOT = Path(r'C:\Users\LENOVO\Downloads\Urahackathon2026\the-2nd-ura-hackathon')
    WORK_DIR  = Path('.')

# Phase 2: private_test.csv + sample_submission_private.csv (nam trong phase2_dataset/...)
TEST_CSV   = _find_first(COMP_ROOT, ['private_test.csv', 'test_private.csv',
                                     'public_test.csv', 'test.csv'])
SAMPLE_CSV = _find_first(COMP_ROOT, ['sample_submission_private.csv', 'sample_submission.csv',
                                     'sample_submission_public.csv'])
TRAIN_CSV  = _find_first(COMP_ROOT, ['train_labels.csv', 'train.csv', 'train_label.csv'])
# Uu tien tim anh trong CUNG nhanh voi file test (phase2_dataset/...) -> tranh anh train
_test_base = TEST_CSV.parent if TEST_CSV is not None else COMP_ROOT
IMAGES_DIR = (_find_images_dir(_test_base) or _find_images_dir(COMP_ROOT) or COMP_ROOT)

print('Auto-discovery:')
print(f'  COMP_ROOT  : {COMP_ROOT}  (exists={COMP_ROOT.exists()})')
print(f'  TEST_CSV   : {TEST_CSV}')
print(f'  SAMPLE_CSV : {SAMPLE_CSV}')
print(f'  TRAIN_CSV  : {TRAIN_CSV}')
print(f'  IMAGES_DIR : {IMAGES_DIR}')

assert TEST_CSV is not None and TEST_CSV.exists(), (
    'Khong tim thay test csv duoi %s. Kiem tra lai dataset da Add chua.' % COMP_ROOT)

OUTPUT_CSV     = WORK_DIR / 'submission.csv'
CHECKPOINT_CSV = WORK_DIR / 'checkpoint.csv'

test_df = pd.read_csv(TEST_CSV)
train_labels_df = pd.read_csv(TRAIN_CSV, keep_default_na=False) if (TRAIN_CSV and TRAIN_CSV.exists()) else None

# ===== Auto-detect schema (vong moi co the tach brand_name / product_name) =====
def _detect_col(df, *cands):
    if df is None: return None
    low = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in low: return low[c.lower()]
    return None

# Chuan hoa cot ID cua test ve 'image_id' (phong khi vong moi dat ten khac)
_TEST_ID = _detect_col(test_df, 'image_id', 'id', 'img_id', 'image', 'filename', 'file_name')
if _TEST_ID and _TEST_ID != 'image_id':
    test_df = test_df.rename(columns={_TEST_ID: 'image_id'})

BRAND_COL   = _detect_col(train_labels_df, 'brand_name', 'brand', 'thuong_hieu')
PRODUCT_COL = _detect_col(train_labels_df, 'product_name', 'product', 'san_pham')
OCR_COL     = _detect_col(train_labels_df, 'ocr_text', 'ocr', 'text')
HAS_BRAND_COL = BRAND_COL is not None

# Sample submission quyet dinh thu tu / ten cot dau ra (auto-align format moi)
if SAMPLE_CSV is not None and SAMPLE_CSV.exists():
    _sample_head = pd.read_csv(SAMPLE_CSV, keep_default_na=False, nrows=1)
    SUBMISSION_COLS = list(_sample_head.columns)
else:
    SUBMISSION_COLS = ['image_id', 'ocr_text', 'brand_name', 'product_name']
SUB_HAS_BRAND   = any(c.lower() in ('brand_name','brand') for c in SUBMISSION_COLS)
SUB_HAS_PRODUCT = any(c.lower() in ('product_name','product') for c in SUBMISSION_COLS)

_all_imgs = list(IMAGES_DIR.glob('*')) if IMAGES_DIR.is_dir() else []
n_imgs = sum(1 for p in _all_imgs if p.suffix.lower() in ['.jpg','.jpeg','.png'])
print(f'COMP_ROOT  : {COMP_ROOT}  (exists={COMP_ROOT.exists()})')
print(f'IMAGES_DIR : {IMAGES_DIR}  (exists={IMAGES_DIR.is_dir()}, {n_imgs} imgs)')
if _all_imgs: print(f'  -> Vi du  : {_all_imgs[0].name}')
print(f'Test set   : {len(test_df):,} rows')
print(f'Submission cols : {SUBMISSION_COLS}')
if train_labels_df is not None:
    print(f'Train cols : {list(train_labels_df.columns)}')
    print(f'  -> brand_col={BRAND_COL!r} product_col={PRODUCT_COL!r} ocr_col={OCR_COL!r}')
    if PRODUCT_COL is not None:
        has_prod = (train_labels_df[PRODUCT_COL].astype(str).str.strip()!='').sum()
        print(f'Train data : {len(train_labels_df):,} rows, co product: {has_prod:,}')
    if HAS_BRAND_COL:
        has_brand = (train_labels_df[BRAND_COL].astype(str).str.strip()!='').sum()
        print(f'           : co brand_name: {has_brand:,} (train-driven brand discovery BAT)')
    else:
        print('           : KHONG co cot brand_name -> se suy ra brand tu nhan/regex (legacy mode)')
else:
    print('Train data : Khong tim thay!')


Auto-discovery:
  COMP_ROOT  : /kaggle/input/competitions/the-2nd-ura-hackathon  (exists=True)
  TEST_CSV   : /kaggle/input/competitions/the-2nd-ura-hackathon/phase2_dataset/phase2_dataset/private_test.csv
  SAMPLE_CSV : /kaggle/input/competitions/the-2nd-ura-hackathon/phase2_dataset/phase2_dataset/sample_submission_private.csv
  TRAIN_CSV  : /kaggle/input/competitions/the-2nd-ura-hackathon/train_labels.csv
  IMAGES_DIR : /kaggle/input/competitions/the-2nd-ura-hackathon/phase2_dataset/phase2_dataset/images/images
COMP_ROOT  : /kaggle/input/competitions/the-2nd-ura-hackathon  (exists=True)
IMAGES_DIR : /kaggle/input/competitions/the-2nd-ura-hackathon/phase2_dataset/phase2_dataset/images/images  (exists=True, 1202 imgs)
  -> Vi du  : priv_h_0242.jpg
Test set   : 1,202 rows
Submission cols : ['image_id', 'ocr_text', 'brand_name', 'product_name']
Train cols : ['image_id', 'ocr_text', 'product_name']
  -> brand_col=None product_col='product_name' ocr_col='ocr_text'
Train data : 4,892 rows, 

In [3]:
IMAGE_PATH_INDEX = {}
for p in IMAGES_DIR.glob('*'):
    if p.suffix.lower() in ('.jpg', '.jpeg', '.png'):
        IMAGE_PATH_INDEX[p.stem] = str(p)
        IMAGE_PATH_INDEX[p.name] = str(p)

print(f'Indexed {len(IMAGE_PATH_INDEX):,} image paths')
print('Mẫu image_id:', test_df['image_id'].head(5).tolist())
_missing = [fid for fid in test_df['image_id'].head(5) if str(fid) not in IMAGE_PATH_INDEX]
print('Thiếu (nếu có):', _missing)

Indexed 2,404 image paths
Mẫu image_id: ['priv_h_0001', 'priv_h_0002', 'priv_h_0003', 'priv_h_0004', 'priv_h_0005']
Thiếu (nếu có): []


In [4]:
import unicodedata

# ==================== UTILITY ====================

def _fold_ascii(s):
    s = str(s).replace('\u0111','d').replace('\u0110','D').casefold()
    s = unicodedata.normalize('NFD', s)
    return ''.join(c for c in s if unicodedata.category(c)!='Mn')

def strip_diacritics(text):
    return _fold_ascii(text)


# ==================== SOCIAL NOISE FILTER ====================
SOCIAL_NOISE_WORDS = frozenset({
    'tiktok','capcut','instagram','reels','facebook','youtube',
    'follow','like','share','comment','subscribe','livestream',
    'fyp','duet','stitch','viral','trending','video','clip','news',
})

OCR_TYPO_MAP = (
    (r'canfuc([o0])', r'canfoc\1'),
    (r'vinamill\b', 'vinamilk'),
    (r'vinamik\b',  'vinamilk'),
    (r'halong\b',   'ha long'),
    (r'patê',       'pate'),
)

def clean_social_ocr(text):
    import re
    text = re.sub(r'@[\w\.]+|#\w+|https?://\S+|www\.\S+', ' ', str(text))
    text = re.sub(r'\b\d{1,2}:\d{2}(:\d{2})?\b', ' ', text)
    text = re.sub(r'(\d\s*){5,}', ' ', text)
    for pat, repl in OCR_TYPO_MAP:
        text = re.sub(pat, repl, text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


PRODUCT_MIN_OCR_CHARS = 4

def _is_ocr_noise_only(text):
    text = str(text or '').strip()
    if len(text) < PRODUCT_MIN_OCR_CHARS: return True
    tokens = re.sub(
        r'[^\w\s\+\u00c0-\u1ef9\u0111\u0110]', ' ', text.lower()
    ).split()
    if not tokens: return True
    meaningful = [
        t for t in tokens
        if len(t)>=3 and t not in SOCIAL_NOISE_WORDS
        and not t.startswith('@') and 'www' not in t
    ]
    return len(meaningful)==0


# ==================== BRAND RULES v4.0 ====================
# Format: (regex, canonical_brand, [product_line_keywords])
BRAND_RULES = [
    # --- HALONG CANFOCO + PATE COT DEN ---
    (r'ha\s*long\s*canf[ou]c[o0].*pate.*c[\u1ed9o]t|c[\u1ed9o]t\s*[d\u0111][\u00e8e]n.*ha\s*long\s*canf[ou]c[o0]',
     'Ha Long Canfoco Pate C\u1ed9t \u0110\u00e8n', []),
    (r'ha\s*long\s*canf[ou]c[o0].*pate|canf[ou]c[o0].*pate\s*c[\u1ed9o]t|pate\s*c[\u1ed9o]t.*canf[ou]c[o0]',
     'Ha Long Canfoco Pate', []),
    (r'ha\s*long\s*canf[ou]c[o0]|halong\s*canf[ou]c[o0]|canfuc[o0]|canfood|\bcanf[ou]c[o0]\b',
     'Ha Long Canfoco', []),
    # THEM: 'canf' (OCR cat ngan)
    (r'halong\s*canf\b|ha\s*long\s*canf\b', 'Ha Long Canfoco', []),

    # --- DO HOP HA LONG ---
    (r'[d\u0111][\u1ed3o]\s*h[\u1ed9o]p\s*h[\u1ea1a]\s*long|do\s*hop\s*ha\s*long',
     '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long', []),
    # THEM: hophalong (OCR dich lien)
    (r'hophalong|hop\s*ha\s*long', '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long', []),

    # --- PATE COT DEN HAI PHONG ---
    (r'pate\s*c[\u1ed9o]t\s*[d\u0111][\u00e8e]n|pate\s*cot\s*den|c[\u1ed9o]t\s*[d\u0111][\u00e8e]n\s*h[\u1ea3a]i\s*ph[\u00f2o]ng',
     'Pate C\u1ed9t \u0110\u00e8n H\u1ea3i Ph\u00f2ng', []),
    # THEM: cotcen, cotd variants
    (r'cotc[e]n.*hai|cot\s*c[e]n.*hai|cotd.*hai\s*ph[\u00f2o]ng',
     'Pate C\u1ed9t \u0110\u00e8n H\u1ea3i Ph\u00f2ng', []),
    # THEM: bien the OCR [ate/fate/jate + cotcen/cotden
    (r'[\[fjh]?at[e\u00ea].*cot.*d[e\u00ea\u00e8n]|cotc[e]n|cot\s*c[e]n\b|\bcotd[e\u00ea]?\b',
     'Pate C\u1ed9t \u0110\u00e8n', []),

    # --- STANDARD BRANDS ---
    (r'pate\s*h[\u1ea1a]\s*long|h[\u1ea1a]\s*long\s*pate', 'Ha Long Canfoco Pate', []),
    (r'vinamilk|vinamill|vinamik', 'Vinamilk',
     ['flex','adm gold','sure','canxi','colosbaby','colos baby','ong tho','\u00f4ng th\u1ecd','dielac','grow']),
    (r'th\s*true|thtrue|th\s*true\s*milk', 'TH True Milk',
     ['true yogurt','grow','school milk','true butter']),
    (r'dutch\s*lady|c\u00f4\s*g\u00e1i|dutchlady', 'Dutch Lady',
     ['grow','complete','canxi','mom']),
    (r'nutifood|\bnuti\b', 'Nutifood', ['growplus','grow plus','pedia','iq']),
    (r'\bensure\b', 'Abbott Ensure', ['gold','original','plus']),
    (r'pediasure', 'Abbott PediaSure', []),
    (r'similac', 'Abbott Similac', []),
    (r'glucerna', 'Abbott Glucerna', []),
    (r'\bmilo\b', 'Nestl\u00e9 Milo', []),
    (r'nestle|nestl\u00e9', 'Nestl\u00e9',
     ['milo','coffee mate','carnation','nestea','nan','s\u1eefa b\u1ed9t']),
    (r'aptamil', 'Aptamil', ['first infant formula','profutura','gold']),
    (r'\bhipp\b', 'HiPP', ['combiotic','organic']),
    (r'\bfriso\b', 'Friso', ['gold','comfort','prestige']),
    (r'\bmeiji\b', 'Meiji', ['growing','step']),
    (r'nan\s*(optipro|opti\s*pro|infinipro|infini\s*pro|supremepro|supreme\s*pro|a2|ha)\b',
     'Nestl\u00e9 NAN', []),
    (r'sua\s*nan\b|nestle\s*nan|nestl\u00e9\s*nan', 'Nestl\u00e9 NAN', []),
    # THEM: dulac/dielac (OCR sai Dielac Vinamilk)
    (r'\bdulac\b|\bdielac\b', 'Vinamilk Dielac', []),
    (r'ba\s*vi\b|bavi\b|ba\s*v\u00ec', 'Ba V\u00ec', ['gold']),
    (r'lothamilk', 'Lothamilk', ['canxi']),
    (r'yomost', 'Yomost', []),
    (r'dalat\s*milk|\u0111\u00e0\s*l\u1ea1t', '\u0110\u00e0 L\u1ea1t Milk', []),
    (r'\bkun\b|kun\s*milk', 'Kun', ['chocolate','strawberry']),
    (r'\bfami\b', 'Fami', ['canxi','kid']),
    (r'anlene', 'Anlene', ['gold','concentrate']),
    (r'\banchor\b', 'Anchor', ['butter','cream']),
    (r'vissan', 'Vissan',
     ['pate heo','pate ga','pate g\u00e0','xuc xich','x\u00fac x\u00edch','lap xuong','l\u1ea1p x\u01b0\u1edfng']),
    (r'\bhafi\b', 'Hafi', ['pate']),
    (r'ba\s*huan|ba\s*hu\u00e2n', 'Ba Hu\u00e2n', ['pate']),
    (r'san\s*ha\b|san\s*h\u00e0', 'San H\u00e0', ['pate']),
    (r'\bcp\b|c\.p\.', 'CP', ['pate','x\u00fac x\u00edch']),
    (r'long\s*bien|long\s*bi\u00ean', 'Long Bi\u00ean', ['pate']),
    (r'highlands?\s*coffee', 'Highlands Coffee',
     ['tra sen vang','tra vai','banh mi que','americano']),
    (r'the\s*coffee\s*house', 'The Coffee House', ['tra phuc kien']),
    (r'nhan\s*hoa\s*foods?', 'Nh\u00e2n H\u00f2a Foods', ['pate']),
    (r'\bacnes\b', 'Acnes', ['vitamin cleanser']),
    (r'\bpate\b|pat\u00ea', 'Pate', []),
]

_SUB_NAMES = {
    'flex':'Flex','adm gold':'ADM Gold','sure':'Sure','canxi':'Canxi',
    'colosbaby':'ColosBaby','colos baby':'Colos Baby',
    'ong tho':'\u00d4ng Th\u1ecd','\u00f4ng th\u1ecd':'\u00d4ng Th\u1ecd',
    'dielac':'Dielac','grow':'Grow','true yogurt':'True Yogurt',
    'school milk':'School Milk','true butter':'True Butter',
    'growplus':'GrowPlus+','grow plus':'GrowPlus+','pedia':'Pedia','iq':'IQ',
    'gold':'Gold','original':'Original','plus':'Plus',
    'milo':'Milo','coffee mate':'Coffee Mate','nan':'NAN','s\u1eefa b\u1ed9t':'S\u1eefa B\u1ed9t',
    'first infant formula':'First Infant Formula','profutura':'Profutura',
    'combiotic':'Combiotic','organic':'Organic',
    'growing':'Growing','step':'Step',
    'comfort':'Comfort','prestige':'Prestige',
    'pate heo':'Pate Heo','pate ga':'Pate G\u00e0','pate g\u00e0':'Pate G\u00e0',
    'xuc xich':'X\u00fac X\u00edch','x\u00fac x\u00edch':'X\u00fac X\u00edch',
    'lap xuong':'L\u1ea1p X\u01b0\u1edfng','l\u1ea1p x\u01b0\u1edfng':'L\u1ea1p X\u01b0\u1edfng',
    'pate':'Pate','x\u00fac x\u00edch':'X\u00fac X\u00edch',
    'tra sen vang':'Tr\u00e0 Sen V\u00e0ng','tra vai':'Tr\u00e0 V\u1ea3i',
    'banh mi que':'B\u00e1nh M\u00ec Que','americano':'Americano',
    'tra phuc kien':'Tr\u00e0 Ph\u00fac Ki\u1ebfn',
    'vitamin cleanser':'Vitamin Cleanser','concentrate':'Concentrate',
    'butter':'Butter','cream':'Cream','chocolate':'Chocolate','strawberry':'Strawberry',
    'kid':'Kid','kid+':'Kid+',
}


def _normalize_for_rules(text):
    tl = text.lower().replace('pat\u00ea','pate')
    tl = re.sub(r'vina[jilm1]{0,1}milk', 'vinamilk', tl, flags=re.IGNORECASE)
    tl = re.sub(r'canfuc([o0])', r'canfoc\1', tl, flags=re.IGNORECASE)
    tl = re.sub(r'halong\b', 'ha long', tl, flags=re.IGNORECASE)
    return tl


# --- TANAHIVER6: anti-hallucination token filter ---
def _build_brand_vocab():
    vocab = set()
    for _pat, brand, lines in BRAND_RULES:
        for w in brand.split(): vocab.add(w.casefold())
        for line in lines:
            for w in line.title().split(): vocab.add(w.casefold())
    return frozenset(vocab)

_BRAND_VOCAB = _build_brand_vocab()
_BRAND_VOCAB_FOLDED = frozenset(_fold_ascii(t) for t in _BRAND_VOCAB)


def _ocr_word_set(text):
    tl = _normalize_for_rules(text)
    return {_fold_ascii(w) for w in re.findall(r'[^\W\d_]+', tl, flags=re.UNICODE) if w}


def _token_in_ocr(tok, ocr_text, ocr_words):
    tf = _fold_ascii(tok)
    if tf in ocr_words: return True
    compact = _fold_ascii(ocr_text).replace(' ','')
    if len(tf)>=2 and tf in compact: return True
    return any(len(tf)>=2 and tf in ow for ow in ocr_words)


def _assign_readable_brand_tokens(ocr_text, candidate):
    '''Chi giu token thuoc brand list VA doc duoc tu ocr_text (chong hallucination)'''
    if not candidate or not candidate.strip(): return ''
    ocr_words = _ocr_word_set(ocr_text)
    kept = []
    for tok in candidate.split():
        if _fold_ascii(tok) not in _BRAND_VOCAB_FOLDED: continue
        if _token_in_ocr(tok, ocr_text, ocr_words): kept.append(tok)
    return ' '.join(kept)


def extract_by_rules(text):
    # FIX v5: rule da neo theo regex tren OCR nen TIN CAY -> tra ve nhan day du,
    # KHONG di qua _assign_readable_brand_tokens (von lam roi 'Pate Cot Den' -> 'Cot').
    if not text: return ''
    tl = _normalize_for_rules(text)
    matched = ''
    for pattern, brand, lines in BRAND_RULES:
        if re.search(pattern, tl, re.IGNORECASE):
            for line in lines:
                if re.search(line, tl, re.IGNORECASE):
                    sub = _SUB_NAMES.get(line, line.title())
                    matched = f'{brand} {sub}'.strip()
                    break
            if not matched:
                matched = brand
            break
    return matched


def post_process_prediction(pred, ocr_text):
    '''Upgrade 'Pate' generic -> cu the hon bang cach re-scan OCR'''
    if not pred: return pred
    nd = _fold_ascii(ocr_text) if ocr_text else ''
    if pred == 'Pate':
        if re.search(r'hai\s*ph[o]ng', nd):
            if re.search(r'cot|cotcen|cotd|col\s*d', nd):
                return 'Pate C\u1ed9t \u0110\u00e8n H\u1ea3i Ph\u00f2ng'
        if any(re.search(p, nd) for p in [r'cot\s*d[e]n',r'\bcotd[e]?\b',r'cotcen',r'cot\s*cen']):
            return 'Pate C\u1ed9t \u0110\u00e8n'
        if re.search(r'ha\s*long|halong', nd):
            return '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long'
    if pred == 'Nestl\u00e9':
        if re.search(r'\bnan\b|nan\s*opti|nan\s*infini|nan\s*supreme', nd):
            return 'Nestl\u00e9 NAN'
        if re.search(r'\bmilo\b', nd):
            return 'Nestl\u00e9 Milo'
    if pred == 'Vinamilk':
        if re.search(r'dulac|dielac', nd):
            return 'Vinamilk Dielac'
    return pred
# ==================== PRODUCT NAME NORMALIZATION ====================
_PRODUCT_CANON_MAP = None

def _build_product_canon_map():
    """Gop cac bien the chinh ta/dau cau cua cung 1 san pham (VD: 'côt' vs 'cột',
    'patê' vs 'pate') bang fold-key, chon spelling pho bien nhat trong train data
    lam chinh tac. Manh hon Title Case don thuan vi xu ly duoc ca loi dau cau."""
    global _PRODUCT_CANON_MAP
    if _PRODUCT_CANON_MAP is not None:
        return _PRODUCT_CANON_MAP
    _PRODUCT_CANON_MAP = {}
    try:
        _pcol = PRODUCT_COL if ('PRODUCT_COL' in globals() and PRODUCT_COL) else 'product_name'
        names = train_labels_df[_pcol].astype(str).str.strip()
        names = names[names != '']
        counts = Counter(unicodedata.normalize('NFC', n) for n in names)
        groups = {}
        for name, cnt in counts.items():
            key = _fold_ascii(name)
            groups.setdefault(key, []).append((cnt, name))
        for key, variants in groups.items():
            # uu tien: nhieu mau nhat -> neu hoa, chon ten day du dau nhat
            variants.sort(key=lambda x: (-x[0], -len(x[1])))
            _, best_name = variants[0]
            _PRODUCT_CANON_MAP[key] = best_name.title()
    except Exception:
        pass
    return _PRODUCT_CANON_MAP


def normalize_product_name(name):
    """Chuan hoa nhan san pham: NFC + khoang trang + gop bien the
    viet hoa/thuong VA chinh ta/dau cau ve 1 ten chinh tac duy nhat."""
    if not name:
        return ''
    name = unicodedata.normalize('NFC', str(name).strip())
    name = re.sub(r'\s+', ' ', name)
    if not name:
        return ''
    canon_map = _build_product_canon_map()
    key = _fold_ascii(name)
    return canon_map.get(key, name.title())

# Smoke test
_tests = [
    ('HA LONG CANFOCO Pate C\u1ed9t \u0110\u00e8n', 'Ha Long Canfoco Pate C\u1ed9t \u0110\u00e8n'),
    ('HA LONG CANFOCO pate cot den Hai Phong',    'Ha Long Canfoco Pate'),
    ('HALONG CANFUCO j TIkTok',                   'Ha Long Canfoco'),
    ('\u0110\u1ed3 h\u1ed9p H\u1ea1 Long ISO22000', '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long'),
    ('Pate c\u1ed9t \u0111\u00e8n',               'Pate C\u1ed9t \u0110\u00e8n'),
    ('Vinamilk Flex 180ml',                       'Vinamilk Flex'),
    ('MILO Nestle chocolate 3 in 1',              'Nestl\u00e9 Milo'),
    ('Pate Heo Vissan 170g',                      'Vissan Pate Heo'),
    ('TH True Milk tuoi tiet trung',              'TH True Milk'),
    ('Dutch Lady Grow+ 900g',                     'Dutch Lady Grow'),
    ('No brand text',                             ''),
    ('tiktok capcut viral',                       ''),
    ('Pate COTDEN 130 tan thit lon',              'Pate'),  # cotden in text but maybe
]
print('Rules v4.0 | %d rules | Smoke test:' % len(BRAND_RULES))
ok_n=0
for txt, exp in _tests:
    r = extract_by_rules(txt)
    ok = (exp=='' and r=='') or (exp!='' and exp.split()[0].lower() in r.lower())
    if ok: ok_n+=1
    print('  [%s] %-45s => %s' % ('OK' if ok else '??', txt[:45], repr(r)[:40]))
print('  Score: %d/%d' % (ok_n, len(_tests)))


# ============================================================================
# v5.0 BRAND / PRODUCT SPLIT LAYER (vong moi: 2 cot brand_name + product_name)
# ============================================================================

# Brand thuan (de longest-prefix match khi tach nhan gop). Dai dat truoc.
KNOWN_BRANDS = [
    'Ha Long Canfoco', 'Pate C\u1ed9t \u0110\u00e8n', 'TH True Milk', 'Dutch Lady',
    'Highlands Coffee', 'The Coffee House', 'Nh\u00e2n H\u00f2a Foods', 'H\u1ea1 Long',
    'Abbott', 'Nestl\u00e9', 'Vinamilk', 'Nutifood', 'Aptamil', 'HiPP', 'Friso',
    'Meiji', 'Ba V\u00ec', 'Lothamilk', 'Yomost', '\u0110\u00e0 L\u1ea1t Milk', 'Kun',
    'Fami', 'Anlene', 'Anchor', 'Vissan', 'Hafi', 'Ba Hu\u00e2n', 'San H\u00e0',
    'CP', 'Long Bi\u00ean', 'Acnes',
]

# Best-guess tach cac nhan gop kho cua vong cu -> (brand, product)
MERGED_SPLIT = {
    'Ha Long Canfoco Pate C\u1ed9t \u0110\u00e8n': ('Ha Long Canfoco', 'Pate C\u1ed9t \u0110\u00e8n'),
    'Ha Long Canfoco Pate':                       ('Ha Long Canfoco', 'Pate'),
    'Ha Long Canfoco':                            ('Ha Long Canfoco', ''),
    '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long':         ('H\u1ea1 Long', '\u0110\u1ed3 H\u1ed9p'),
    'Pate C\u1ed9t \u0110\u00e8n H\u1ea3i Ph\u00f2ng': ('Pate C\u1ed9t \u0110\u00e8n', 'H\u1ea3i Ph\u00f2ng'),
    'Pate C\u1ed9t \u0110\u00e8n':                 ('Pate C\u1ed9t \u0110\u00e8n', ''),
    'Pate':                                       ('', 'Pate'),
    'Vinamilk Dielac':                            ('Vinamilk', 'Dielac'),
    'Nestl\u00e9 NAN':                            ('Nestl\u00e9', 'NAN'),
    'Nestl\u00e9 Milo':                           ('Nestl\u00e9', 'Milo'),
}

# Token product qua chung chung -> khong cho dung 1 minh (chong loi 'Cot'/'Hop' le)
GENERIC_PRODUCT_TOKENS = frozenset({
    'cot','hop','milk','sua','gold','plus','new','lon','can','hu','ml','g','kg',
    'den','sen','do','ha','long',
})

MIN_BRAND_SUPPORT   = 2   # brand moi tu train can >= so mau nay
MIN_BRAND_ALIAS_LEN = 4   # alias ngan hon -> bo qua (tranh dung 'cp','ps')

# Registry: folded_alias -> canonical brand. Seed tu KNOWN_BRANDS, mo rong tu train.
BRAND_REGISTRY = {}
def _register_brand(canonical):
    canonical = unicodedata.normalize('NFC', str(canonical).strip())
    if not canonical: return
    aliases = set()
    f = _fold_ascii(canonical)
    aliases.add(f)
    aliases.add(f.replace(' ', ''))
    for a in list(aliases):
        if len(a.replace(' ','')) >= MIN_BRAND_ALIAS_LEN or ' ' in a:
            BRAND_REGISTRY[a] = canonical

for _b in KNOWN_BRANDS:
    _register_brand(_b)

_BRAND_CANON_FOLD = {}   # fold(brand) -> canonical casing (uu tien train)

def build_brand_knowledge_from_train():
    """Train-driven discovery: hoc brand moi (vd 'Dove') tu cot brand_name,
    co kiem soat support toi thieu. Khong tu bia: chi dua brand co trong train."""
    if not (HAS_BRAND_COL and train_labels_df is not None):
        return 0
    s = train_labels_df[BRAND_COL].astype(str).map(lambda x: unicodedata.normalize('NFC', x.strip()))
    s = s[s != '']
    vc = s.value_counts()
    added = 0
    for brand, cnt in vc.items():
        canon = brand if brand.isupper() or brand.istitle() else brand
        _BRAND_CANON_FOLD.setdefault(_fold_ascii(brand), brand)
        if cnt >= MIN_BRAND_SUPPORT:
            _register_brand(brand)
            added += 1
    return added


def normalize_brand_name(name):
    if not name: return ''
    name = re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(name).strip()))
    if not name: return ''
    f = _fold_ascii(name)
    if f in _BRAND_CANON_FOLD: return _BRAND_CANON_FOLD[f]
    if f in BRAND_REGISTRY:    return BRAND_REGISTRY[f]
    fc = f.replace(' ', '')
    if fc in BRAND_REGISTRY:   return BRAND_REGISTRY[fc]
    return name


def detect_brand_in_ocr(ocr_text):
    """Quet alias brand (KNOWN + train) trong OCR. Yeu cau co EVIDENCE that su
    trong OCR -> giam hallucination. Tra canonical brand dai/khop nhat.
    An toan: alias 1 tu phai khop NGUYEN TU (tranh false positive ngan)."""
    if not ocr_text: return ''
    folded = _fold_ascii(_normalize_for_rules(ocr_text))
    words = set(folded.split())
    compact = folded.replace(' ', '')
    best = ''
    for alias, canonical in BRAND_REGISTRY.items():
        a = alias.replace(' ', '')
        if len(a) < MIN_BRAND_ALIAS_LEN: continue
        if ' ' in alias:
            hit = alias in folded                       # alias nhieu tu
        else:
            hit = (a in words) or (len(a) >= 6 and a in compact)  # alias 1 tu
        if hit and len(canonical) > len(best):
            best = canonical
    return best


def _is_generic_product(product):
    if not product: return True
    f = _fold_ascii(product).strip()
    if len(f) < 2: return True
    toks = f.split()
    if len(toks) == 1 and toks[0] in GENERIC_PRODUCT_TOKENS: return True
    return False


def split_brand_product(merged):
    """Tach nhan gop (vong cu) -> (brand_name, product_name)."""
    if not merged: return '', ''
    nm = re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(merged).strip()))
    if nm in MERGED_SPLIT:
        b, p = MERGED_SPLIT[nm]
        return normalize_brand_name(b), p
    nf = _fold_ascii(nm)
    for brand in sorted(KNOWN_BRANDS, key=len, reverse=True):
        bf = _fold_ascii(brand)
        if nf == bf:
            return normalize_brand_name(brand), ''
        if nf.startswith(bf + ' '):
            return normalize_brand_name(brand), nm[len(brand):].strip()
    return '', nm


# =====================================================================
# v6.0 ADDITIONS from pipeline.ipynb
# =====================================================================

# --- Bilingual English product words (on-pack labels) ---
ENGLISH_PRODUCT_WORDS = {
    # Cosmetics & beauty
    'maybelline','mascara','hyper','curl','waterproof','gel','volume',
    'matte','cream','lotion','serum','cleanser','shampoo','conditioner',
    'eyeliner','eyebrow','lipstick','lipbalm','foundation','powder',
    'blush','skincare','active','formula','natural','organic','extract',
    'sunblock','sunscreen','moisture','uv','protect','smoothie',
    # General on-pack English
    'original','premium','classic','style','design','brand','product',
    'made','in','vietnam','usa','korea','japan','quality','best',
    'super','ultra','mega','extreme','pro','max','smart','light','dark',
    'color','black','white','red','blue','green','ml','g','kg','size',
    'pack','set','limit','edition','free','soft','smooth','fresh','cool',
    'dry','wet','hot','warm',
}

# --- Additional brand normalization map (merge into BRAND_REGISTRY) ---
BRAND_NORMALIZATION_MAP = {
    'ha long':      'Ha Long Canfoco',
    'canfoco':      'Ha Long Canfoco',
    'do hop ha long': '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long',
    'dove':         'Dove',
    'vinamilk':     'Vinamilk',
    'vissan':       'Vissan',
    'nestle':       'Nestl\u00e9',
    'nescafe':      'Nescafe',
    'omo':          'Omo',
    'knorr':        'Knorr',
    'maybelline':   'Maybelline',
}
# Register extra brands into BRAND_REGISTRY
for _alias, _canonical in BRAND_NORMALIZATION_MAP.items():
    _register_brand(_canonical)

# --- Variant & promo patterns ---
VARIANT_PATTERN = r'\d+\s*(g|ml|kg|l|gr|gx|h\u1ed9p|g\u00f3i|chai|l\u1ed1c|lon|c\u00e1i|vi\u00ean|h\u0169)\s*(x\s*\d+)?|\b(combo|l\u1ed1c|v\u1ec9|set)\s*\d+'
PROMO_PATTERN   = r'\b(gi\u1ea3m|giam)\s*\d+%|\b\d+%\s*(off|gi\u1ea3m|giam)?|\bmua\s*\d+\s*t\u1eb7ng\s*\d+'
PROMO_KEYWORDS  = {'ch\u00ednh h\u00e3ng','\u0111\u1ed9c quy\u1ec1n','freeship',
                   'cam k\u1ebft','uy t\u00edn','qu\u00e0 t\u1eb7ng','m\u1edbi','new','official'}
DESCRIPTION_INDICATORS = [
    '\u0111\u1ea7u c\u1ecd','cho mi','d\u1ea1ng gel','c\u00f4ng th\u1ee9c',
    'h\u00e0ng mi','th\u1ea5m n\u01b0\u1edbc','ch\u1ea3i t\u1eebng',
    'nh\u1eb9 m\u01b0\u1ee3t','ch\u1ed1ng tr\u00f4i','c\u00f4ng vut',
    'su\u1ed1t nhi\u1ec1u','si\u00eau m\u1ecbn','l\u00e0n da','d\u01b0\u1ee1ng \u1ea9m',
]


def _remove_accents(s):
    import unicodedata as _ud
    nfd = _ud.normalize('NFD', s)
    return ''.join(c for c in nfd if not _ud.combining(c)).replace('\u0110','D').replace('\u0111','d')


def bilingual_post_processor(text):
    """Fix bilingual OCR errors: accented English words and digit/letter swaps."""
    import unicodedata as _ud
    words = text.split()
    out = []
    for word in words:
        clean = re.sub(r'^\W+|\W+$', '', word)
        # Fix digit→letter in English-only tokens (e.g. m0del→model)
        if re.match(r'^[a-zA-Z0-9]+$', clean):
            temp = clean.replace('0','o').replace('1','i')
            if temp.lower() in ENGLISH_PRODUCT_WORDS:
                out.append(word.replace(clean, temp)); continue
        # Fix accented English words (e.g. máscara→mascara)
        if any(c in 'àáảãạằắặầấẩậèéẻẽẹềếểệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵđ' for c in clean.lower()):
            stripped = _remove_accents(clean)
            if stripped.lower() in ENGLISH_PRODUCT_WORDS:
                if clean.istitle():   stripped = stripped.capitalize()
                elif clean.isupper(): stripped = stripped.upper()
                else:                 stripped = stripped.lower()
                out.append(word.replace(clean, stripped)); continue
        out.append(word)
    return ' '.join(out)


def is_negative_line(text):
    """True if this OCR line is noise (date, email, URL, pure number)."""
    patterns = [
        r'\d{2,4}[-/\.]\d{2}[-/\.]\d{2,4}',
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        r'www\.|http[s]?://|\.com|\.vn',
        r'^\d+$',
    ]
    if any(re.search(p, text) for p in patterns): return True
    neg_kw = {'ng\u00e0y s\u1ea3n xu\u1ea5t','th\u00e0nh ph\u1ea7n',
              'ingredients','ch\u1ec9 ti\u00eau'}
    return any(kw in text.lower() for kw in neg_kw)


import difflib as _difflib

def fuzzy_match_brand(text, normalization_map=None, threshold=0.75):
    """Match brand in text using substring + difflib fuzzy.
    Returns (found: bool, canonical_brand: str)."""
    if normalization_map is None:
        normalization_map = BRAND_NORMALIZATION_MAP
    t_low = re.sub(r'\W+', '', text.lower())
    for variant, canonical in normalization_map.items():
        if re.sub(r'\W+', '', variant) in t_low:
            return True, canonical
    for word in text.lower().split():
        cw = re.sub(r'\W+', '', word)
        for variant, canonical in normalization_map.items():
            vc = re.sub(r'\W+', '', variant)
            if _difflib.SequenceMatcher(None, cw, vc).ratio() >= threshold:
                return True, canonical
    return False, None

print('v6.0 helpers loaded: bilingual_post_processor, fuzzy_match_brand, BRAND_NORMALIZATION_MAP')


Rules v4.0 | 46 rules | Smoke test:
  [OK] HA LONG CANFOCO Pate Cột Đèn                  => 'Ha Long Canfoco Pate Cột Đèn'
  [OK] HA LONG CANFOCO pate cot den Hai Phong        => 'Ha Long Canfoco Pate Cột Đèn'
  [OK] HALONG CANFUCO j TIkTok                       => 'Ha Long Canfoco'
  [OK] Đồ hộp Hạ Long ISO22000                       => 'Đồ Hộp Hạ Long'
  [OK] Pate cột đèn                                  => 'Pate Cột Đèn Hải Phòng'
  [OK] Vinamilk Flex 180ml                           => 'Vinamilk Flex'
  [OK] MILO Nestle chocolate 3 in 1                  => 'Nestlé Milo'
  [OK] Pate Heo Vissan 170g                          => 'Vissan Pate Heo'
  [OK] TH True Milk tuoi tiet trung                  => 'TH True Milk'
  [OK] Dutch Lady Grow+ 900g                         => 'Dutch Lady Grow'
  [OK] No brand text                                 => ''
  [OK] tiktok capcut viral                           => ''
  [OK] Pate COTDEN 130 tan thit lon                  => 'Pate Cột Đèn Hải Phòng'
  

In [5]:
import os, threading, numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image as _PIL_Image

OCR_CONF_THRESHOLD = 0.35
_ocr_worker_local  = threading.local()
NUM_WORKERS        = max(1, min(CPU_THREADS, 4))


# =====================================================================
# PaddleOCR — DETECTION ONLY (get bounding boxes)
# =====================================================================
def _latin_dict_path():
    import paddleocr
    return str(
        Path(paddleocr.__file__).resolve().parent
        / 'ppocr' / 'utils' / 'dict' / 'latin_dict.txt'
    )

def _paddle_model_dirs():
    whl = Path.home() / '.paddleocr' / 'whl'
    return (
        whl / 'det' / 'ch' / 'ch_PP-OCRv4_det_infer',
        whl / 'rec' / 'latin' / 'latin_PP-OCRv3_rec_infer',
    )

def _paddle_model_ready(d):
    d = Path(d)
    return (d/'inference.pdmodel').is_file() and (d/'inference.pdiparams').is_file()

_PADDLE_URLS = (
    'https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_infer.tar',
    'https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/latin_PP-OCRv3_rec_infer.tar',
)

def _prefetch_models():
    from ppocr.utils.network import maybe_download
    det_dir, rec_dir = _paddle_model_dirs()
    pending = [(str(det_dir), _PADDLE_URLS[0]), (str(rec_dir), _PADDLE_URLS[1])]
    pending = [(d,u) for d,u in pending if not _paddle_model_ready(d)]
    if not pending: return det_dir, rec_dir
    print(f'Prefetch {len(pending)} PaddleOCR model(s)...')
    with ThreadPoolExecutor(max_workers=min(NUM_WORKERS,2)) as pool:
        futs = {pool.submit(maybe_download,d,u): Path(d).name for d,u in pending}
        for fut in as_completed(futs):
            fut.result(); print(f'  ready: {futs[fut]}')
    return det_dir, rec_dir

def _make_paddle_det():
    """Return PaddleOCR in detection-only mode (rec disabled via rec_algorithm=None trick)."""
    from paddleocr import PaddleOCR
    det_dir, rec_dir = _prefetch_models()
    return PaddleOCR(
        use_gpu=False,
        use_angle_cls=True,
        show_log=False,
        enable_mkldnn=True,
        lang='latin',
        det_model_dir=str(det_dir),
        rec_model_dir=str(rec_dir),
        rec_char_dict_path=_latin_dict_path(),
    )


# =====================================================================
# VietOCR — RECOGNITION on crops
# =====================================================================
_vietocr_predictor = None
_vietocr_lock      = threading.Lock()

def _get_vietocr():
    global _vietocr_predictor
    if _vietocr_predictor is None:
        with _vietocr_lock:
            if _vietocr_predictor is None:
                from vietocr.tool.predictor import Predictor
                from vietocr.tool.config import Cfg
                import torch
                torch.set_num_threads(max(1, CPU_THREADS // 2))
                cfg = Cfg.load_config_from_name('vgg_seq2seq')
                cfg['device'] = 'cpu'
                cfg['predictor']['beamsearch'] = False   # faster on CPU
                _vietocr_predictor = Predictor(cfg)
                print('VietOCR vgg_seq2seq loaded (CPU)')
    return _vietocr_predictor

def _recognize_crop(crop_np):
    """Use VietOCR to read one crop (numpy array RGB)."""
    try:
        pil = _PIL_Image.fromarray(crop_np)
        return _get_vietocr().predict(pil)
    except Exception as e:
        return ''

def _crop_box(pil_img, box, pad=3):
    """Crop a bounding box from PIL image with padding."""
    xs = [p[0] for p in box]; ys = [p[1] for p in box]
    w, h = pil_img.size
    xmin = max(0, int(min(xs)) - pad)
    ymin = max(0, int(min(ys)) - pad)
    xmax = min(w, int(max(xs)) + pad)
    ymax = min(h, int(max(ys)) + pad)
    return np.array(pil_img.crop((xmin, ymin, xmax, ymax)))


# =====================================================================
# Image utilities
# =====================================================================
def preprocess(img, max_dim=1280):
    w, h = img.size
    if max(w,h) > max_dim:
        r = max_dim / max(w,h)
        img = img.resize((int(w*r), int(h*r)), _PIL_Image.LANCZOS)
    img = ImageEnhance.Contrast(img).enhance(1.35)
    return img.filter(ImageFilter.SHARPEN)

def postprocess_ocr(text):
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    if not tokens: return ''
    deduped = [tokens[0]]
    for tok in tokens[1:]:
        if tok.lower() != deduped[-1].lower(): deduped.append(tok)
    return ' '.join(deduped)

def load_image(image_id):
    path = IMAGE_PATH_INDEX.get(str(image_id))
    if path is None or not Path(path).exists(): return None
    try: return _PIL_Image.open(str(path)).convert('RGB')
    except: return None


# =====================================================================
# Full OCR: PaddleOCR detect -> VietOCR recognize
# =====================================================================
_engine_create_lock = threading.Lock()

def _run_full_ocr(pil_img):
    """
    1. Preprocess image
    2. PaddleOCR: detect bounding boxes
    3. Sort boxes top-to-bottom, left-to-right
    4. Crop each box
    5. VietOCR: recognize each crop
    6. Bilingual post-process each recognized text
    7. Join into one OCR string
    """
    img = preprocess(pil_img)
    img_np_bgr = np.array(img)[:,:,::-1]

    # --- Detect ---
    try:
        raw = ocr_engine.ocr(img_np_bgr, cls=True)
    except Exception:
        return ''
    if not raw or not raw[0]:
        return ''

    # Filter by confidence, sort by position
    items = []
    for box, (text_raw, conf) in raw[0]:
        if conf > OCR_CONF_THRESHOLD:
            yx = box[0]
            items.append((yx[1], yx[0], box))
    items.sort()

    if not items:
        return ''

    # --- Crop + VietOCR recognize ---
    texts = []
    for _, _, box in items:
        crop = _crop_box(img, box)
        recognized = _recognize_crop(crop)
        if recognized and recognized.strip():
            cleaned = bilingual_post_processor(recognized)
            texts.append(cleaned)

    return postprocess_ocr(' '.join(texts))


def _ocr_image_only(image_id, _images_dir_str):
    """Worker: load image and run full OCR (detect+recognize)."""
    engine = getattr(_ocr_worker_local, 'engine', None)
    if engine is None:
        os.environ['OMP_NUM_THREADS'] = '1'
        os.environ['MKL_NUM_THREADS'] = '1'
        _ocr_worker_local.engine = _make_paddle_det()
    img = load_image(image_id)
    if img is None: return image_id, ''
    # Use per-worker paddle engine for detection
    global ocr_engine
    _orig = ocr_engine
    try:
        ocr_engine = _ocr_worker_local.engine
        result = _run_full_ocr(img)
    finally:
        ocr_engine = _orig
    return image_id, result


print('Loading PaddleOCR (CPU detect) + VietOCR (CPU recognize)...')
ocr_engine = _make_paddle_det()
print(f'PaddleOCR detector OK | conf>{OCR_CONF_THRESHOLD} | {NUM_WORKERS} workers')
print('Loading VietOCR (lazy, first call will download model)...')
_ = _get_vietocr()   # eager load so first batch is not slow
print('VietOCR OK')

print('\nSmoke test on first image...')
_fid  = test_df['image_id'].iloc[0]
_img  = load_image(_fid)
if _img:
    _ocr = _run_full_ocr(_img)
    print(f'  image_id : {_fid}')
    print(f'  ocr_text : {_ocr[:80]}{"..." if len(_ocr)>80 else ""}')
else:
    print(f'  Warning: {_fid} not found')


Loading PaddleOCR (CPU, latin_PP-OCRv3)...


PLEASE USE OMP_NUM_THREADS WISELY.


Prefetch 2 PaddleOCR model(s)...
download https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_infer.tar to /root/.paddleocr/whl/det/ch/ch_PP-OCRv4_det_infer/ch_PP-OCRv4_det_infer.tar
download https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/latin_PP-OCRv3_rec_infer.tar to /root/.paddleocr/whl/rec/latin/latin_PP-OCRv3_rec_infer/latin_PP-OCRv3_rec_infer.tar


 68%|██████▊   | 6771/9930 [00:23<00:01, 2531.30it/s]

  ready: ch_PP-OCRv4_det_infer


100%|██████████| 9930/9930 [00:24<00:00, 411.72it/s] 


  ready: latin_PP-OCRv3_rec_infer
download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2138/2138 [00:15<00:00, 139.64it/s]


PaddleOCR OK | conf>0.35 | 4 parallel worker(s)

Smoke test on first image...
  image_id : priv_h_0001
  ocr_text : BIOESSENCE MASK PRE MADECASSOSIDE Cantata fiber BIOESSENCEM 30ml KEYSHU BIOES MA...


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline as SkPipeline


class ProductPredictor:
    def __init__(self, min_class=3, prob_thr=0.60, max_feat=3000):
        self.min_class=min_class; self.prob_thr=prob_thr; self.max_feat=max_feat
        self._has_clf=self._prod_clf=None; self.fitted=False

    def fit(self, df, rule_fn):
        df=df.copy()
        df['ocr_text']=df['ocr_text'].astype(str).str.strip()
        df['product_name']=df['product_name'].astype(str).str.strip()
        self._rule_fn=rule_fn
        # Binary: has product?
        self._has_clf=SkPipeline([
            ('tfidf', TfidfVectorizer(analyzer='char_wb',ngram_range=(2,4),max_features=self.max_feat,min_df=2)),
            ('clf',   LogisticRegression(max_iter=400, class_weight='balanced')),
        ])
        self._has_clf.fit(df['ocr_text'], (df['product_name']!='').astype(int))
        # Multi-class: which product?
        pos=df[(df['ocr_text']!='')&(df['product_name']!='')]
        keep=pos['product_name'].value_counts()
        keep=keep[keep>=self.min_class].index
        pos=pos[pos['product_name'].isin(keep)]
        self._prod_clf=SkPipeline([
            ('tfidf', TfidfVectorizer(analyzer='char_wb',ngram_range=(2,4),max_features=self.max_feat,min_df=2)),
            ('clf',   LogisticRegression(max_iter=400, class_weight='balanced')),
        ])
        if len(pos):
            self._prod_clf.fit(pos['ocr_text'], pos['product_name'])
        self.fitted=True
        print(f'  ProductPredictor: {len(df):,} rows, {pos["product_name"].nunique() if len(pos) else 0} classes')
        return self

    def predict(self, ocr_text):
        ocr_text=str(ocr_text or '').strip()
        if not ocr_text or _is_ocr_noise_only(ocr_text): return ''
        ruled=self._rule_fn(ocr_text)
        if ruled: return post_process_prediction(ruled, ocr_text)
        if not self.fitted or self._has_clf is None: return ''
        proba=self._has_clf.predict_proba([ocr_text])[0]
        classes=list(self._has_clf.classes_)
        if 1 not in classes or proba[classes.index(1)] < self.prob_thr: return ''
        pred=str(self._prod_clf.predict([ocr_text])[0])
        result=_assign_readable_brand_tokens(ocr_text, pred)
        return post_process_prediction(result, ocr_text) if result else ''


# v5.0: ProductPredictor (nhan gop) la LEGACY, KHONG dung trong pipeline moi.
# Pipeline v5 dung BrandClassifier + ProductClassifier (cell sau). Giu class de tham khao.
product_predictor = None
print('ProductPredictor (legacy merged) DISABLED -> dung dual predictor v5.0 o cell sau.')


ProductPredictor (legacy merged) DISABLED -> dung dual predictor v5.0 o cell sau.


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================================
# v5.0 DUAL PREDICTOR: BrandClassifier + ProductClassifier (+ KNN fallback)
#   - Hoc tu train theo schema moi (brand_name / product_name) hoac legacy.
#   - Train-driven discovery: brand moi (vd 'Dove') duoc hoc tu train co kiem soat.
# ============================================================================

def _build_training_frame():
    """Chuan hoa train -> DataFrame voi cot _ocr / _brand / _product (auto schema)."""
    if train_labels_df is None:
        return None
    df = train_labels_df.copy()
    oc = OCR_COL if (OCR_COL and OCR_COL in df.columns) else ('ocr_text' if 'ocr_text' in df.columns else None)
    df['_ocr'] = df[oc].astype(str).str.strip() if oc else ''
    if HAS_BRAND_COL:
        df['_brand'] = df[BRAND_COL].astype(str).str.strip().map(normalize_brand_name)
        if PRODUCT_COL is not None:
            df['_product'] = df[PRODUCT_COL].astype(str).str.strip().map(normalize_product_name)
        else:
            df['_product'] = ''
    else:
        merged = df[PRODUCT_COL].astype(str).str.strip() if PRODUCT_COL is not None else pd.Series(['']*len(df))
        bp = merged.map(split_brand_product)
        df['_brand'] = bp.map(lambda t: normalize_brand_name(t[0]))
        df['_product'] = bp.map(lambda t: normalize_product_name(t[1]))
    return df[df['_ocr'] != '']


def _pipe(mf=6000, C=1.5):
    return Pipeline([
        ('tf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=mf, sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=800, class_weight='balanced', C=C, solver='lbfgs')),
    ])


class BrandClassifier:
    """Brand nang 0.4 -> uu tien evidence (alias trong OCR), classifier la fallback bao thu."""
    def __init__(self, min_cls=MIN_BRAND_SUPPORT, thr=0.62):
        self.min_cls=min_cls; self.thr=thr; self._clf=None; self.fitted=False
    def fit(self, df):
        pos = df[df['_brand'] != '']
        vc = pos['_brand'].value_counts()
        keep = vc[vc >= self.min_cls].index
        pos = pos[pos['_brand'].isin(keep)]
        if pos['_brand'].nunique() >= 2:
            self._clf = _pipe(mf=5000).fit(pos['_ocr'], pos['_brand'])
            self.fitted = True
        print('  BrandClassifier: %d rows, %d brands (support>=%d).' % (len(pos), len(keep), self.min_cls))
        return self
    def predict(self, ocr_text):
        ocr_text = str(ocr_text or '').strip()
        if not ocr_text:
            return ''
        b = detect_brand_in_ocr(ocr_text)   # 1) evidence-based (chong hallucination)
        if b:
            return b
        if self.fitted and self._clf is not None:   # 2) classifier fallback
            try:
                proba = self._clf.predict_proba([ocr_text])[0]
                i = int(proba.argmax())
                if proba[i] >= self.thr:
                    return normalize_brand_name(str(self._clf.classes_[i]))
            except Exception:
                pass
        return ''


class ProductClassifier:
    """Product nang 0.25 -> cho phep doan khi du tin cay, co chan token generic."""
    def __init__(self, min_cls=2, thr=0.45, gate_thr=0.40):
        self.min_cls=min_cls; self.thr=thr; self.gate_thr=gate_thr
        self._gate=None; self._clf=None; self.fitted=False
    def fit(self, df):
        yg = (df['_product'] != '').astype(int)
        if yg.nunique() >= 2:
            self._gate = _pipe(mf=6000).fit(df['_ocr'], yg)
        pos = df[df['_product'] != '']
        vc = pos['_product'].value_counts()
        pos = pos[pos['_product'].isin(vc[vc >= self.min_cls].index)]
        if pos['_product'].nunique() >= 2:
            self._clf = _pipe(mf=6000).fit(pos['_ocr'], pos['_product'])
            self.fitted = True
        print('  ProductClassifier: %d pos, %d products (support>=%d).' % (
            int(yg.sum()), pos['_product'].nunique(), self.min_cls))
        return self
    def predict(self, ocr_text):
        ocr_text = str(ocr_text or '').strip()
        if not ocr_text or not self.fitted:
            return ''
        try:
            if self._gate is not None:
                gp = self._gate.predict_proba([ocr_text])[0]
                cl = list(self._gate.classes_)
                if 1 in cl and gp[cl.index(1)] < self.gate_thr:
                    return ''
            proba = self._clf.predict_proba([ocr_text])[0]
            i = int(proba.argmax())
            if proba[i] < self.thr:
                return ''
            p = normalize_product_name(str(self._clf.classes_[i]))
            return '' if _is_generic_product(p) else p
        except Exception:
            return ''


class BrandKNN:
    """Fallback similarity: bo phieu brand & product tu cac mau train gan nhat."""
    def __init__(self, k=5, thr=0.55):
        self.k=k; self.thr=thr; self._tf=None; self._mat=None
        self._brands=[]; self._products=[]; self.fitted=False
    def fit(self, df):
        self._tf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=6000, sublinear_tf=True)
        self._mat = self._tf.fit_transform(df['_ocr'])
        self._brands = df['_brand'].tolist()
        self._products = df['_product'].tolist()
        self.fitted = True
        print('  BrandKNN: %d vectors.' % len(self._brands))
        return self
    def _vote(self, labels, idx, sims):
        votes = [labels[i] for i in idx if sims[i] >= self.thr and labels[i]]
        return Counter(votes).most_common(1)[0][0] if votes else ''
    def predict(self, ocr_text):
        if not self.fitted or not ocr_text or len(str(ocr_text).strip()) < 5:
            return '', ''
        try:
            sv = self._tf.transform([ocr_text])
            sims = cosine_similarity(sv, self._mat).flatten()
            idx = sims.argsort()[::-1][:self.k]
            return self._vote(self._brands, idx, sims), self._vote(self._products, idx, sims)
        except Exception:
            return '', ''


brand_clf = product_clf = brand_knn = None
_train_frame = _build_training_frame()
if _train_frame is not None and len(_train_frame):
    _added = build_brand_knowledge_from_train()
    print('Brand knowledge: +%d brand tu train (registry=%d alias).' % (_added, len(BRAND_REGISTRY)))
    print('Training BrandClassifier...')
    brand_clf = BrandClassifier().fit(_train_frame)
    print('Training ProductClassifier...')
    product_clf = ProductClassifier().fit(_train_frame)
    print('Training BrandKNN...')
    brand_knn = BrandKNN().fit(_train_frame)
    print('Dual predictor ready.')
else:
    print('No train data -> brand/product classifiers OFF (rules-only).')


Brand knowledge: +0 brand tu train (registry=42 alias).
Training BrandClassifier...
  BrandClassifier: 1106 rows, 14 brands (support>=2).
Training ProductClassifier...
  ProductClassifier: 2603 pos, 206 products (support>=2).
Training BrandKNN...
  BrandKNN: 3973 vectors.
Dual predictor ready.


In [8]:
def _strip_brand_prefix(brand, product):
    """Bo token brand bi lap o dau product (vd brand 'Vinamilk', product 'Vinamilk Dielac')."""
    if not (brand and product):
        return product
    bf = _fold_ascii(brand)
    pf = _fold_ascii(product)
    if pf == bf:
        return ''
    if pf.startswith(bf + ' '):
        return product[len(brand):].strip()
    return product


def predict_labels(ocr_text, image_id=None):
    """v5.0: tra ve (brand_name, product_name) rieng biet.
    Thu tu uu tien: rules high-conf -> brand/product classifier -> KNN -> rong."""
    ocr_text = str(ocr_text or '').strip()
    if _is_ocr_noise_only(ocr_text):
        return '', ''   # case D: OCR chi la noise -> de rong, khong bi phat

    brand = product = ''

    # 1) Rules (precision cao) -> nhan gop -> tach brand/product
    merged = extract_by_rules(ocr_text)
    if merged:
        merged = post_process_prediction(merged, ocr_text)
        brand, product = split_brand_product(merged)

    # 2a) Brand evidence-based alias scan (luon chay, ke ca khi khong co train)
    if not brand:
        brand = detect_brand_in_ocr(ocr_text)

    # 2b) Brand fallback: classifier (OCR nhieu, khong khop alias)
    # [STEP 2c v6.0] Fuzzy brand match from normalization map
    if not brand:
        _fz_found, _fz_brand = fuzzy_match_brand(ocr_text, BRAND_NORMALIZATION_MAP)
        if _fz_found and _fz_brand:
            brand = normalize_brand_name(_fz_brand)
    if not brand and brand_clf is not None:
        brand = brand_clf.predict(ocr_text)

    # 3) Product fallback: classifier
    if not product and product_clf is not None:
        product = product_clf.predict(ocr_text)

    # 4) KNN fallback cho field con thieu
    if (not brand or not product) and brand_knn is not None:
        kb, kp = brand_knn.predict(ocr_text)
        if not brand and kb:
            brand = normalize_brand_name(kb)
        if not product and kp and not _is_generic_product(kp):
            product = normalize_product_name(kp)

    # 5) Guard cuoi cung
    brand = normalize_brand_name(brand) if brand else ''
    if product:
        product = normalize_product_name(product)
        product = _strip_brand_prefix(brand, product)
        if _is_generic_product(product):
            product = ''
    return brand, product


# ===== Smoke tests (vong moi: brand + product tach rieng) =====
# Dang ky tam 'Dove' de minh hoa train-driven discovery (offline khong co train Dove)
_register_brand('Dove')

# (ocr, expected_brand_substr | None, expected_substr_in_combined | None)
#   - eb kiem tra rieng cot brand_name
#   - ec kiem tra cum xuat hien trong brand+product (tranh fix cung field do taxonomy)
_P = [
    ('HALONG CANFOCO Pate C\u1ed9t \u0110\u00e8n', 'Ha Long Canfoco', 'Pate'),
    ('ate Jate Cotcen 130 tan thit lon',           None,             'Pate C\u1ed9t \u0110\u00e8n'),  # KHONG duoc ra 'Cot' le
    ('HIGHLANDS COFFEE tra sen vang',              'Highlands Coffee', None),
    ('tiktok viral capcut fyp news',               '',               ''),                 # case D
    ('HiPP Combiotic organic milk',                'HiPP',           None),
    ('Nestl\u00e9 NAN OPTIpro 0-6',                'Nestl\u00e9',    'NAN'),
    ('Dove Smoothie t\u1ea9y da ch\u1ebft',        'Dove',           None),                # brand moi ngoai regex
    ('Vinamilk Dielac so 1',                        'Vinamilk',       'Dielac'),
]
print('Pipeline v5.0 brand/product test:')
ok_n = 0
for txt, eb, ec in _P:
    b, p = predict_labels(txt)
    combined = (b + ' ' + p).strip()
    ok = True
    if eb is not None:
        ok = ok and ((eb == '' and b == '') or (eb != '' and eb.lower() in b.lower()))
    if ec is not None:
        ok = ok and ((ec == '' and combined == '') or (ec != '' and ec.lower() in combined.lower()))
    # Bao dam KHONG BAO GIO tra product generic 1 token le (loi cu 'Cot')
    if p and _is_generic_product(p):
        ok = False
    if ok:
        ok_n += 1
    print('  [%s] %-42s => brand=%-16s product=%s' % (
        'OK' if ok else 'FAIL', txt[:42], repr(b)[:16], repr(p)[:30]))
print('  Score: %d/%d' % (ok_n, len(_P)))


Pipeline v5.0 brand/product test:
  [OK] HALONG CANFOCO Pate Cột Đèn                => brand='Ha Long Canfoco product='Pate Cột Đèn'
  [OK] ate Jate Cotcen 130 tan thit lon           => brand='Pate Cột Đèn'   product=''
  [OK] HIGHLANDS COFFEE tra sen vang              => brand='Highlands Coffe product='Trà Sen Vàng'
  [OK] tiktok viral capcut fyp news               => brand=''               product=''
  [OK] HiPP Combiotic organic milk                => brand='HiPP'           product='Combiotic'
  [OK] Nestlé NAN OPTIpro 0-6                     => brand='Nestlé'         product='Nan'
  [OK] Dove Smoothie tẩy da chết                  => brand='Dove'           product=''
  [OK] Vinamilk Dielac so 1                       => brand='Vinamilk'       product='Dielac'
  Score: 8/8


In [9]:
# ============================================================================
# v5.0 EVALUATOR theo cong thuc moi:
#   Diem = 0.4 * F1_brand + 0.35 * (1 - CER) + 0.25 * F1_product
#   - F1 tinh theo token (fold dau, lowercase).
#   - Case D (GT rong & pred rong) -> F1 = 1.0 (khong bi phat).
#   - CER khong do duoc offline (can OCR du doan vs GT) -> uoc luong theo ASSUMED_CER.
# ============================================================================

def _token_f1(pred, gt):
    pt  = set(_fold_ascii(pred).split()) if pred else set()
    gt_ = set(_fold_ascii(gt).split())   if gt   else set()
    if not pt and not gt_:
        return 1.0   # case D
    if not pt or not gt_:
        return 0.0   # case C (mot ben rong)
    tp = len(pt & gt_)
    if tp == 0:
        return 0.0
    p = tp / len(pt)
    r = tp / len(gt_)
    return 2 * p * r / (p + r)


def evaluate_pipeline(df=None, n=500, assumed_cer=0.25):
    if df is None:
        df = _train_frame
    if df is None or not len(df):
        return None
    if len(df) > n:
        df = df.sample(n, random_state=42)
    sb = sp = exact_b = exact_p = ev = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Eval'):
        ocr = str(row['_ocr']).strip()
        gt_b = str(row['_brand']).strip()
        gt_p = str(row['_product']).strip()
        pb, pp = predict_labels(ocr)
        sb += _token_f1(pb, gt_b)
        sp += _token_f1(pp, gt_p)
        if _fold_ascii(pb) == _fold_ascii(gt_b): exact_b += 1
        if _fold_ascii(pp) == _fold_ascii(gt_p): exact_p += 1
        ev += 1
    f1b = sb / ev if ev else 0
    f1p = sp / ev if ev else 0
    score = 0.4 * f1b + 0.35 * (1 - assumed_cer) + 0.25 * f1p
    return {'F1_brand': f1b, 'F1_product': f1p,
            'ExactBrand': exact_b/ev if ev else 0,
            'ExactProduct': exact_p/ev if ev else 0,
            'N': ev, 'AssumedCER': assumed_cer, 'EstScore': score}


if _train_frame is not None and len(_train_frame):
    print('Eval %d samples...' % min(500, len(_train_frame)))
    m = evaluate_pipeline(_train_frame, 500)
    print('F1_brand=%.4f  F1_product=%.4f  ExactBrand=%.4f  ExactProduct=%.4f  N=%d'
          % (m['F1_brand'], m['F1_product'], m['ExactBrand'], m['ExactProduct'], m['N']))
    for _cer in (0.0, 0.15, 0.25):
        _s = 0.4*m['F1_brand'] + 0.35*(1-_cer) + 0.25*m['F1_product']
        print('  Est.Score (CER=%.2f): %.4f' % (_cer, _s))
else:
    print('No train data -> skip eval.')


Eval 500 samples...


Eval:   0%|          | 0/500 [00:00<?, ?it/s]

F1_brand=0.5216  F1_product=0.6220  ExactBrand=0.5160  ExactProduct=0.4820  N=500
  Est.Score (CER=0.00): 0.7141
  Est.Score (CER=0.15): 0.6616
  Est.Score (CER=0.25): 0.6266


In [10]:
from joblib import Parallel, delayed

SAVE_EVERY = 50
CHECKPOINT_CSV = WORK_DIR / 'checkpoint.csv'

# Resume tu checkpoint
results, done_ids = [], set()
if CHECKPOINT_CSV.exists():
    ckpt = pd.read_csv(CHECKPOINT_CSV, keep_default_na=False)
    done_ids = set(ckpt['image_id'])
    results = ckpt.to_dict('records')
    print(f'Resume: {len(done_ids):,} done.')
else:
    print('Fresh start.')

pending = [i for i in test_df['image_id'] if i not in done_ids]
print(f'Pending: {len(pending):,} | Workers: {NUM_WORKERS}')

images_dir_str = str(IMAGES_DIR)

import time
errors = 0
t0 = time.perf_counter()

for batch_start in tqdm(range(0, len(pending), SAVE_EVERY), desc='OCR batches'):
    batch = pending[batch_start: batch_start+SAVE_EVERY]

    # Parallel OCR
    ocr_pairs = Parallel(n_jobs=NUM_WORKERS, backend='threading', batch_size=4)(
        delayed(_ocr_image_only)(iid, images_dir_str) for iid in batch
    )

    # Brand + Product prediction (main process)
    for image_id, ocr_text in ocr_pairs:
        try:
            cleaned = clean_social_ocr(ocr_text)
            brand, product = predict_labels(cleaned, image_id)
            results.append({
                'image_id': image_id,
                'ocr_text': ocr_text,
                'brand_name': brand,
                'product_name': product,
            })
        except Exception as e:
            results.append({'image_id': image_id, 'ocr_text': '',
                            'brand_name': '', 'product_name': ''})
            errors += 1

    # Save checkpoint
    pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8')

elapsed = time.perf_counter() - t0
print(f'\nDone {len(results)} imgs in {elapsed/60:.1f} min ({elapsed/max(len(results),1):.2f}s/img)')
n_ocr   = sum(1 for r in results if str(r.get('ocr_text','')).strip())
n_brand = sum(1 for r in results if str(r.get('brand_name','')).strip())
n_prod  = sum(1 for r in results if str(r.get('product_name','')).strip())
print(f'OCR ok: {n_ocr} | Brand: {n_brand} | Product: {n_prod} | Errors: {errors}')


Fresh start.
Pending: 1,202 | Workers: 4


OCR batches:   0%|          | 0/25 [00:00<?, ?it/s]


Done 1202 imgs in 12.1 min (0.60s/img)
OCR ok: 1113 | Brand: 60 | Product: 29 | Errors: 0


In [11]:
import csv as _csv

# ===== Build submission khop dung schema cua sample_submission.csv (vong moi) =====
sample = pd.read_csv(SAMPLE_CSV, keep_default_na=False)
sample_cols = list(sample.columns)

# Map ten cot dau ra -> field noi bo
def _pick(cols, *cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in low: return low[c.lower()]
    return None

ID_OUT      = _pick(sample_cols, 'image_id', 'id') or 'image_id'
OCR_OUT     = _pick(sample_cols, 'ocr_text', 'ocr', 'text')
BRAND_OUT   = _pick(sample_cols, 'brand_name', 'brand')
PRODUCT_OUT = _pick(sample_cols, 'product_name', 'product')

res_df = pd.DataFrame(results)
sub = pd.DataFrame({ID_OUT: res_df['image_id']})
if OCR_OUT:     sub[OCR_OUT]     = res_df.get('ocr_text', '')
if BRAND_OUT:   sub[BRAND_OUT]   = res_df.get('brand_name', '')
if PRODUCT_OUT: sub[PRODUCT_OUT] = res_df.get('product_name', '')

# Cot text: strip, thay rong = ' ' (tranh null), bo newline/tab
text_cols = [c for c in (OCR_OUT, BRAND_OUT, PRODUCT_OUT) if c]
for col in text_cols:
    sub[col] = sub[col].fillna('').astype(str).str.replace(r'[\n\t\r]', ' ', regex=True).str.strip()
    sub.loc[sub[col] == '', col] = ' '

# Dam bao du & dung thu tu cot theo sample
for c in sample_cols:
    if c not in sub.columns:
        sub[c] = ' '
sub = sub[sample_cols]

checks = {
    'AC1 rows':       len(sub) == len(sample),
    'AC2 no extra':   not bool(set(sub[ID_OUT]) - set(sample[ID_OUT])),
    'AC3 no miss':    not bool(set(sample[ID_OUT]) - set(sub[ID_OUT])),
    'AC4 no dup':     not sub[ID_OUT].duplicated().any(),
    'AC5 no null':    not sub.isnull().any().any(),
    'AC6 no newline': not any(sub[c].str.contains(r'[\n\t\r]', regex=True, na=False).any() for c in text_cols),
    'AC7 col order':  list(sub.columns) == sample_cols,
}
ok = all(checks.values())
for k, v in checks.items():
    print('  [%s] %s' % ('PASS' if v else 'FAIL', k))

if ok:
    sub = sub.set_index(ID_OUT).reindex(sample[ID_OUT]).reset_index()
    sub = sub[sample_cols]
    sub.to_csv(OUTPUT_CSV, index=False, encoding='utf-8', quoting=_csv.QUOTE_ALL)
    print('Saved: %s (%.1f KB) | cols=%s' % (OUTPUT_CSV, OUTPUT_CSV.stat().st_size/1024, sample_cols))
    if BRAND_OUT:
        nb = (sub[BRAND_OUT].str.strip() == '').sum()
        print('No brand   : %d (%.1f%%)' % (nb, nb/len(sub)*100))
        top_b = sub[sub[BRAND_OUT].str.strip() != ''][BRAND_OUT].str.strip().value_counts().head(10)
        print('Top brands:')
        for nm, cnt in top_b.items(): print('  %5dx %s' % (cnt, nm))
    if PRODUCT_OUT:
        npd = (sub[PRODUCT_OUT].str.strip() == '').sum()
        print('No product : %d (%.1f%%)' % (npd, npd/len(sub)*100))
        top_p = sub[sub[PRODUCT_OUT].str.strip() != ''][PRODUCT_OUT].str.strip().value_counts().head(10)
        print('Top products:')
        for nm, cnt in top_p.items(): print('  %5dx %s' % (cnt, nm))
else:
    print('[ERROR] Some ACs failed!')


  [PASS] AC1 rows
  [PASS] AC2 no extra
  [PASS] AC3 no miss
  [PASS] AC4 no dup
  [PASS] AC5 no null
  [PASS] AC6 no newline
  [PASS] AC7 col order
Saved: /kaggle/working/submission.csv (140.1 KB) | cols=['image_id', 'ocr_text', 'brand_name', 'product_name']
No brand   : 1142 (95.0%)
Top brands:
     17x Abbott
      7x Aptamil
      6x Dove
      6x Hạ Long
      5x Pate Cột Đèn
      3x CP
      3x Vinamilk
      3x Ha Long Canfoco
      2x TH True Milk
      2x Nutifood
No product : 1173 (97.6%)
Top products:
      7x Similac
      5x Đồ Hộp
      4x Pediasure
      3x Profutura
      2x Hải Phòng
      2x Ensure Gold
      1x Dielac
      1x Combiotic
      1x Ensure
      1x Pate Cột Đèn
